# Eclipse — HalfKAv2 NNUE Eval Training

Trains the Eclipse chess engine's NNUE on Lichess Stockfish depth-22 cp-eval labels.

**Inputs**
* Kaggle Dataset: `eclipse-partial` containing `eval_training.txt.gz`.
  Lines are `<fen>;<score_cp>` — Stockfish depth-22 evaluation, ~685M positions,
  1800+ Elo, 180s+ TC, 20 months (Oct 2024 – May 2026).
* Hardware: GPU T4 ×2 (Settings → Accelerator → GPU T4 x2).
* Settings → Internet: **On**, and a Kaggle Secret named `KAGGLE_API_TOKEN`
  (Add-ons → Secrets) — used to sync `halfkav2.pt` + training position
  (`resume_state.pt`) through the `eclipse-checkpoint` dataset, since
  `/kaggle/working` starts empty on every "Save & Run All".

**Output**
* `/kaggle/working/halfkav2.pt` — latest weights (overwritten each chunk).
* `/kaggle/working/halfkav2_epochN_chunkM.pt` — per-chunk checkpoints.
  All are downloadable from the Output tab of the saved version.
  Convert with `scripts/convert_halfkav2_nnue.py from-torch`.

**Speed**
* AMP (float16), batch 16 384, TF32, cuDNN benchmark, torch.compile, DataParallel.
* ~5–6 h GPU time for 5 epochs over 685M positions.


In [ ]:
# Environment check
import sys, platform
import torch
print(f'python   {platform.python_version()}')
print(f'torch    {torch.__version__}')
print(f'cuda     {torch.cuda.is_available()} ({torch.version.cuda})')
if torch.cuda.is_available():
    print(f'gpu      {torch.cuda.get_device_name(0)}')
    print(f'vram     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Locate eval_training text data anywhere under /kaggle/input/, IF present.
# With premade HKAV2BIN chunks attached (eclipse-chunks-a) the text source is
# NOT required: training reads the chunks directly and the val set is sliced
# from premade chunk 0 (see next cell). DATA_PATH stays None when absent, and
# preprocess_chunk() (the live FEN->bin fallback) is simply never called.
import os
from pathlib import Path

matches = (sorted(Path('/kaggle/input').glob('**/eval_training.txt.gz')) or
           sorted(Path('/kaggle/input').glob('**/eval_training.txt')))
if matches:
    DATA_PATH = matches[0]
    print(f'training text source: {DATA_PATH} ({DATA_PATH.stat().st_size/1e6:.1f} MB)')
else:
    DATA_PATH = None
    print('no eval_training.txt(.gz) under /kaggle/input -- '
          'relying on premade chunks only (live preprocessing disabled)')


In [ ]:
# HalfKAv2 feature indexing -- exact mirror of src/nnue.cpp::feature_index
# and scripts/train_halfkav2.py. Drift here = silently wrong inference.
import gzip, math, random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

FT_IN_FEATURES = 45056          # HalfKAv2: 64 (king sq) * 64 (piece sq) * 11 (piece slots)
# ── net2widernet ──────────────────────────────────────────────────────────
# When NET2WIDER, function-preservingly widen the existing trained net
# (FT 1024->2048, hidden 512->1024 and 128->256, ~2x params) and continue
# training from it instead of starting over. _SRC dims must match the net
# currently in eclipse-checkpoint.
# DEPLOY NOTE: a widened net needs matching C++ engine changes before it loads:
# kFtOutSize in src/nnue.hpp, the accumulator width + SIMD ladders in
# src/nnue.cpp / src/accumulator.hpp, and scripts/convert_halfkav2_nnue.py.
NET2WIDER      = True
FT_OUT_SRC     = 1024           # current deployed net's FT width (kFtOutSize)
L1_OUT_SRC     = 512
L2_OUT_SRC     = 128
FT_OUT         = 2048 if NET2WIDER else FT_OUT_SRC   # keep in sync with kFtOutSize in src/nnue.hpp
L1_OUT         = 1024 if NET2WIDER else L1_OUT_SRC
L2_OUT         = 256  if NET2WIDER else L2_OUT_SRC
L1_IN          = 2 * FT_OUT     # concat of both perspectives
L3_OUT         = 1              # Value head: single logit (sigmoid = win prob)
N_PIECE_SLOTS  = 11

_PIECE_LETTER_TO_TYPE = {'p':1, 'n':2, 'b':3, 'r':4, 'q':5, 'k':6}
_FLIP_RANK = lambda s: s ^ 56

def parse_fen_board(fen_board):
    pieces = []
    sq = 56
    for ch in fen_board:
        if ch == '/':
            sq -= 16
            continue
        if ch.isdigit():
            sq += int(ch)
            continue
        is_white = ch.isupper()
        pt = _PIECE_LETTER_TO_TYPE[ch.lower()]
        pieces.append((sq, 0 if is_white else 1, pt))
        sq += 1
    return pieces

def feature_index(king_sq, piece_sq, pt, piece_is_ours):
    # ours: P=0,N=1,B=2,R=3,Q=4 (own king skipped). theirs: P=5..K=10.
    if piece_is_ours and pt == 6:
        raise ValueError('own king is not a HalfKAv2 feature')
    slot = (pt - 1) if piece_is_ours else (5 + (pt - 1))
    return king_sq * (64 * N_PIECE_SLOTS) + piece_sq * N_PIECE_SLOTS + slot

def fen_to_features(fen):
    parts = fen.split()
    board, stm = parts[0], parts[1]
    pieces = parse_fen_board(board)
    stm_color = 0 if stm == 'w' else 1
    king_sq = [None, None]
    for sq, color, pt in pieces:
        if pt == 6:
            king_sq[color] = sq
    if king_sq[0] is None or king_sq[1] is None:
        raise ValueError(f'FEN missing a king: {fen}')
    indices = [[], []]
    for persp in (0, 1):
        ksq = king_sq[persp] if persp == 0 else _FLIP_RANK(king_sq[persp])
        for sq, color, pt in pieces:
            is_ours = (color == persp)
            if is_ours and pt == 6:
                continue
            psq = sq if persp == 0 else _FLIP_RANK(sq)
            indices[persp].append(feature_index(ksq, psq, pt, is_ours))
    return (indices[0], indices[1]) if stm_color == 0 else (indices[1], indices[0])

In [ ]:
# Dataset helpers — _open_data handles .gz transparently; _parse_line is shared.
# HalfKAv2Dataset: in-memory, for the val set (≤ a few million positions).
# HalfKAv2StreamDataset: reservoir-shuffle streaming, for the full training set.
# HalfKAv2BinaryDataset: memmap random-access, used after preprocessing.
import struct as _struct
import torch
from torch.utils.data import DataLoader, Dataset, IterableDataset
MAX_ACTIVE = 32

def _open_data(path):
    if str(path).endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8')
    return open(path, encoding='utf-8')

def _parse_line(line, cp_scale):
    line = line.strip()
    if not line or line.startswith('#'):
        return None
    parts = line.split(';')
    if len(parts) == 4:
        try:
            fen      = parts[0].strip()
            win_prob = float(parts[1]) + 0.5 * float(parts[2])
        except ValueError:
            return None
    elif len(parts) == 2:
        try:
            fen      = parts[0].strip()
            win_prob = 1.0 / (1.0 + math.exp(-float(parts[1]) / cp_scale))
        except ValueError:
            return None
    else:
        return None
    try:
        us, them = fen_to_features(fen)
    except (KeyError, IndexError, ValueError):
        return None
    import numpy as _np
    us_arr   = _np.full(MAX_ACTIVE, FT_IN_FEATURES, dtype=_np.int32)
    them_arr = _np.full(MAX_ACTIVE, FT_IN_FEATURES, dtype=_np.int32)
    us_arr[:min(len(us),   MAX_ACTIVE)] = us[:MAX_ACTIVE]
    them_arr[:min(len(them), MAX_ACTIVE)] = them[:MAX_ACTIVE]
    return us_arr, them_arr, _np.float32(win_prob)

class HalfKAv2Dataset(Dataset):
    """In-memory dataset. Used for the val set (first val_size lines)."""
    def __init__(self, path, max_lines=None, cp_scale=410.0):
        import numpy as _np
        n = max_lines or self._count(path)
        if n == 0:
            raise RuntimeError(f'no rows in {path}')
        self.us_idx   = _np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        self.them_idx = _np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        self.targets  = _np.zeros((n, 1), dtype=_np.float32)
        i = 0
        with _open_data(path) as f:
            for line in f:
                if i >= n: break
                p = _parse_line(line, cp_scale)
                if p is None: continue
                self.us_idx[i], self.them_idx[i], self.targets[i, 0] = p
                i += 1
        if i < n:
            self.us_idx   = self.us_idx[:i]
            self.them_idx = self.them_idx[:i]
            self.targets  = self.targets[:i]
        print(f'val: loaded {i} positions')
    @staticmethod
    def _count(path):
        n = 0
        with _open_data(path) as f:
            for l in f:
                s = l.strip()
                if s and not s.startswith('#'): n += 1
        return n
    def __len__(self):  return len(self.targets)
    def __getitem__(self, i): return self.us_idx[i], self.them_idx[i], self.targets[i]

class HalfKAv2StreamDataset(IterableDataset):
    """Streaming dataset with reservoir shuffle buffer.
    Memory per worker: shuffle_buffer * (32+32)*4 + shuffle_buffer*4 bytes (~133 MB at 500k).
    max_lines is the TOTAL cap across all workers; each worker gets max_lines // n_workers.
    """
    def __init__(self, path, skip_lines=0, max_lines=None,
                 cp_scale=410.0, shuffle_buffer=500_000):
        self.path, self.skip_lines = path, skip_lines
        self.max_lines, self.cp_scale = max_lines, cp_scale
        self.shuffle_buffer = shuffle_buffer
    def __iter__(self):
        import numpy as _np, random as _rnd, torch as _torch
        wi = _torch.utils.data.get_worker_info()
        wid, nw = (wi.id, wi.num_workers) if wi else (0, 1)
        per_worker_max = ((self.max_lines + nw - 1) // nw
                          if self.max_lines is not None else None)
        buf_us   = _np.full((self.shuffle_buffer, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        buf_them = _np.full((self.shuffle_buffer, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        buf_tgt  = _np.zeros(self.shuffle_buffer, dtype=_np.float32)
        fill = 0; emitted = 0; didx = 0
        with _open_data(self.path) as f:
            if self.skip_lines:
                sk = 0
                for raw in f:
                    if raw.strip() and not raw.strip().startswith('#'):
                        sk += 1
                        if sk >= self.skip_lines: break
            for raw in f:
                if per_worker_max is not None and emitted >= per_worker_max: break
                stripped = raw.strip()
                if not stripped or stripped.startswith('#'): continue
                # Stride check BEFORE parsing: only parse lines this worker owns.
                if didx % nw != wid: didx += 1; continue
                didx += 1
                p = _parse_line(raw, self.cp_scale)
                if p is None: continue
                us_a, th_a, wp = p
                if fill < self.shuffle_buffer:
                    buf_us[fill]=us_a; buf_them[fill]=th_a; buf_tgt[fill]=wp; fill+=1
                else:
                    idx = _rnd.randrange(fill)
                    yield buf_us[idx].copy(), buf_them[idx].copy(), buf_tgt[idx:idx+1].copy()
                    buf_us[idx]=us_a; buf_them[idx]=th_a; buf_tgt[idx]=wp; emitted+=1
        perm = list(range(fill)); _rnd.shuffle(perm)
        for idx in perm:
            if per_worker_max is not None and emitted >= per_worker_max: break
            yield buf_us[idx].copy(), buf_them[idx].copy(), buf_tgt[idx:idx+1].copy()
            emitted += 1

# Binary format constants — must match preprocess_halfkav2.py
_BIN_MAGIC  = b'HKAV2BIN'
_BIN_HSIZE  = 16
_BIN_DTYPE  = np.dtype([('us',  np.uint16, (MAX_ACTIVE,)),
                         ('them',np.uint16, (MAX_ACTIVE,)),
                         ('tgt', np.float16)])

class HalfKAv2BinaryDataset(Dataset):
    """Memmap dataset from preprocess_halfkav2.py. Zero FEN parsing.
    Workers do array indexing only — use DataLoader(shuffle=True)."""
    def __init__(self, path, start=0, end=None):
        with open(path, 'rb') as f:
            if f.read(8) != _BIN_MAGIC: raise ValueError(f'{path}: not HKAV2BIN')
            n_total = _struct.unpack('<q', f.read(8))[0]
        self._mm  = np.memmap(path, dtype=_BIN_DTYPE, mode='r',
                              offset=_BIN_HSIZE, shape=(n_total,))
        self.start = start
        self.end   = n_total if end is None else min(end, n_total)
    def __len__(self): return self.end - self.start
    def __getitem__(self, i):
        r = self._mm[self.start + i]
        return (r['us'].astype(np.int64),
                r['them'].astype(np.int64),
                np.array([float(r['tgt'])], dtype=np.float32))

def collate(batch):
    import numpy as _np
    us   = _np.stack([b[0] for b in batch])
    them = _np.stack([b[1] for b in batch])
    tgt  = _np.stack([b[2] for b in batch])
    return (torch.from_numpy(us).long(),
            torch.from_numpy(them).long(),
            torch.from_numpy(tgt))

In [ ]:
# Model -- float mirror of the C++ inference path in src/nnue.cpp.
# Improvement 1: Quantization-Aware Training (QAT).
#
# Eclipse deploys int16 FT weights and int8 hidden weights. Training in float32
# then converting creates a systematic error: every weight is rounded to the
# nearest quantization step, and the model never saw that rounding during
# training so it can't compensate for it.
#
# QAT inserts fake-quantization nodes (round → dequantize) in the forward pass
# using the Straight-Through Estimator (STE) for gradients. The model sees
# the same discretization error at training time as at inference time, so it
# learns weights that are robust to rounding.
#
# Quantization constants (must match nnue.hpp / convert_halfkav2_nnue.py):
#   FT weights/biases  → int16: stored = round(real * 127),  scale = 1/127
#   Hidden weights     → int8:  stored = round(real * 64),   scale = 1/64
#   Activations (CReLU output) are in [0, 1] = [0, 127/127], discretised to
#   128 levels: scale = 1/127, same as FT.

FT_QUANT     = 127       # kFtQuant in nnue.hpp
WEIGHT_SCALE = 64        # kWeightScale in nnue.hpp

FT_SCALE  = 1.0 / FT_QUANT
FT_LO     = -32768.0 / FT_QUANT
FT_HI     =  32767.0 / FT_QUANT
W_SCALE   = 1.0 / WEIGHT_SCALE
W_LO      = -128.0 / WEIGHT_SCALE   # = -2.0
W_HI      =  127.0 / WEIGHT_SCALE   # ≈ +1.984
ACT_SCALE = 1.0 / FT_QUANT
ACT_LO    = 0.0
ACT_HI    = 1.0


class FakeQuantSTE(torch.autograd.Function):
    """Fake quantization: round to grid in forward, identity (STE) in backward."""
    @staticmethod
    def forward(ctx, x, scale, lo, hi):
        return torch.clamp(torch.round(x / scale) * scale, lo, hi)

    @staticmethod
    def backward(ctx, grad):
        return grad, None, None, None  # STE: pass gradient through unchanged


def fake_quant(x, scale, lo, hi):
    return FakeQuantSTE.apply(x, scale, lo, hi)


class ClippedReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, 0.0, 1.0)


class HalfKAv2Net(nn.Module):
    def __init__(self, qat: bool = False):
        super().__init__()
        self.qat = qat

        # Last embedding row is the padding sentinel: zeroed at init,
        # pinned to zero after every step so padded slots contribute 0.
        self.ft = nn.Embedding(FT_IN_FEATURES + 1, FT_OUT,
                               padding_idx=FT_IN_FEATURES)
        nn.init.uniform_(self.ft.weight, -0.01, 0.01)
        with torch.no_grad():
            self.ft.weight[FT_IN_FEATURES].zero_()
        self.ft_bias = nn.Parameter(torch.zeros(FT_OUT))
        self.l1  = nn.Linear(L1_IN, L1_OUT)
        self.l2  = nn.Linear(L1_OUT, L2_OUT)
        self.l3  = nn.Linear(L2_OUT, L3_OUT)
        self.act = ClippedReLU()

    def forward(self, us_idx, them_idx):
        if self.qat:
            # Fake-quantize FT weights to int16 grid before lookup.
            ft_w_q = fake_quant(self.ft.weight, FT_SCALE, FT_LO, FT_HI)
            ft_b_q = fake_quant(self.ft_bias,   FT_SCALE, FT_LO, FT_HI)
            us   = F.embedding(us_idx,   ft_w_q).sum(dim=1) + ft_b_q
            them = F.embedding(them_idx, ft_w_q).sum(dim=1) + ft_b_q
            # Fake-quantize activations to 128 discrete levels ([0, 127/127]).
            us   = fake_quant(torch.clamp(us,   0.0, 1.0), ACT_SCALE, ACT_LO, ACT_HI)
            them = fake_quant(torch.clamp(them, 0.0, 1.0), ACT_SCALE, ACT_LO, ACT_HI)
        else:
            us   = self.ft(us_idx).sum(dim=1)   + self.ft_bias
            them = self.ft(them_idx).sum(dim=1) + self.ft_bias
            us   = self.act(us)
            them = self.act(them)

        x = torch.cat([us, them], dim=1)

        if self.qat:
            l1_w = fake_quant(self.l1.weight, W_SCALE, W_LO, W_HI)
            x = fake_quant(torch.clamp(F.linear(x, l1_w, self.l1.bias), 0.0, 1.0),
                           ACT_SCALE, ACT_LO, ACT_HI)
            l2_w = fake_quant(self.l2.weight, W_SCALE, W_LO, W_HI)
            x = fake_quant(torch.clamp(F.linear(x, l2_w, self.l2.bias), 0.0, 1.0),
                           ACT_SCALE, ACT_LO, ACT_HI)
            l3_w = fake_quant(self.l3.weight, W_SCALE, W_LO, W_HI)
            return F.linear(x, l3_w, self.l3.bias)
        else:
            x = self.act(self.l1(x))
            x = self.act(self.l2(x))
            return self.l3(x)

    def enable_qat(self):
        self.qat = True

    def disable_qat(self):
        self.qat = False

    def export_state_dict_for_quantization(self):
        sd = {}
        ft_w = self.ft.weight.detach()
        sd['ft.weight'] = ft_w[:FT_IN_FEATURES].T.contiguous()
        sd['ft.bias']   = self.ft_bias.detach()
        sd['l1.weight'] = self.l1.weight.detach()
        sd['l1.bias']   = self.l1.bias.detach()
        sd['l2.weight'] = self.l2.weight.detach()
        sd['l2.bias']   = self.l2.bias.detach()
        sd['l3.weight'] = self.l3.weight.detach()
        sd['l3.bias']   = self.l3.bias.detach()
        return sd


# ── net2widernet: function-preserving widening (Chen et al. 2016) ─────────────
# Operates on the export_state_dict_for_quantization() layout:
#   ft.weight [FT_OUT, FT_IN], ft.bias [FT_OUT],
#   l1.weight [L1_OUT, 2*FT_OUT], l1.bias [L1_OUT],
#   l2.weight [L2_OUT, L1_OUT],   l2.bias [L2_OUT],
#   l3.weight [1, L2_OUT],        l3.bias [1].
# Widening a layer's width q->Q: replicate Q-q existing units (random choice),
# and divide the NEXT layer's incoming weights for each unit by how many times
# it was replicated, so the overall function is preserved. Tiny noise breaks the
# symmetry so the copies can diverge during training.
def _widen_map(n_src, n_dst, device):
    g = torch.arange(n_dst, device=device)
    if n_dst > n_src:
        g[n_src:] = torch.randint(0, n_src, (n_dst - n_src,), device=device)
    counts = torch.bincount(g, minlength=n_src).clamp(min=1).float()
    return g, counts


def net2wider_state_dict(sd, noise=1e-3):
    dev = sd['ft.weight'].device
    g_ft, _ = _widen_map(FT_OUT_SRC, FT_OUT, dev)
    g_l1, _ = _widen_map(L1_OUT_SRC, L1_OUT, dev)
    g_l2, _ = _widen_map(L2_OUT_SRC, L2_OUT, dev)
    # per-DST-unit replication count of the source it maps to
    cnt = lambda g, n: torch.bincount(g, minlength=n).clamp(min=1).float()[g]
    div_ft, div_l1, div_l2 = cnt(g_ft, FT_OUT_SRC), cnt(g_l1, L1_OUT_SRC), cnt(g_l2, L2_OUT_SRC)

    def jitter(t):
        return t + noise * t.abs().mean() * torch.randn_like(t) if noise else t

    out = {}
    # FT width: replicate rows; l1 input cols (both us|them halves) /= count
    out['ft.weight'] = jitter(sd['ft.weight'][g_ft, :].clone())
    out['ft.bias']   = sd['ft.bias'][g_ft].clone()
    l1w  = sd['l1.weight']
    us   = l1w[:, :FT_OUT_SRC][:, g_ft] / div_ft.unsqueeze(0)
    them = l1w[:, FT_OUT_SRC:][:, g_ft] / div_ft.unsqueeze(0)
    l1w  = torch.cat([us, them], dim=1)            # [L1_OUT_SRC, 2*FT_OUT]

    # L1 width: replicate rows; l2 input cols /= count
    out['l1.weight'] = jitter(l1w[g_l1, :].clone())
    out['l1.bias']   = sd['l1.bias'][g_l1].clone()
    l2w = sd['l2.weight'][:, g_l1] / div_l1.unsqueeze(0)   # [L2_OUT_SRC, L1_OUT]

    # L2 width: replicate rows; l3 input cols /= count
    out['l2.weight'] = jitter(l2w[g_l2, :].clone())
    out['l2.bias']   = sd['l2.bias'][g_l2].clone()
    out['l3.weight'] = (sd['l3.weight'][:, g_l2] / div_l2.unsqueeze(0)).clone()
    out['l3.bias']   = sd['l3.bias'].clone()
    return out


In [ ]:
# ── Automated Chunked Preprocessing & Training ──────────────────────────────
# Streams the full dataset in CHUNK_SIZE-position chunks. Each chunk:
#   1. Preprocess → training.bin (memmap binary; deleted after training)
#   2. Train with AMP + DataParallel across both T4 GPUs
#   3. Validate on a FIXED held-out val set (same 200k positions every chunk)
#   4. Save checkpoint to /kaggle/working/ (every 100M positions)
#   5. Repeat until data exhausted, then next epoch.
#
# "Save & Run All" starts each run with an empty /kaggle/working, so neither
# halfkav2.pt nor the training position (epoch/chunk/optimizer/LR-scheduler
# state) survive between runs on their own. CKPT_DATASET on Kaggle is used
# to carry both across runs via the Kaggle API:
#   - ckpt_download() pulls halfkav2.pt + resume_state.pt at startup.
#   - ckpt_upload() pushes a new version after every chunk.
# Requires Settings -> Internet: On and a Kaggle Secret named
# KAGGLE_API_TOKEN (Add-ons -> Secrets). Without it, sync is skipped and
# the run trains from scratch (old behaviour).

import multiprocessing as _mp, struct as _struct2, gzip as _gz2, time as _time2
import shutil, numpy as _np2, os, subprocess as _sp2, json as _json2, tempfile as _tmp2
import torch, torch.nn as _nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader

# ── Configuration ─────────────────────────────────────────────────────────────
CHUNK_SIZE   = 100_000_000   # 100M positions per checkpoint (~13 GB training.bin)
EPOCHS       = 3             # 3 × 1.6B ≈ 5B updates
WORK_DIR     = Path('/kaggle/working')
BIN_PATH     = WORK_DIR / 'training.bin'
VAL_BIN      = WORK_DIR / 'val_fixed.bin'   # fixed val set, created once, never deleted
OUT_PT       = WORK_DIR / 'halfkav2.pt'
RESUME_PT    = WORK_DIR / 'resume_state.pt'  # epoch/chunk/optimizer/scheduler state
CKPT_DATASET = 'simbae11/eclipse-checkpoint'
VAL_SIZE     = 200_000

TRAIN_CFG = dict(
    lr           = 1e-3,    # 1e-3 at batch 16384 matches SF NNUE convention; 2e-3 diverged
    batch_size   = 16384,
    warmup_steps = 500,
    cp_scale     = 300.0,   # Improvement 2: 410 → 300 (25% more gradient in typical range)
    max_cp       = 4000,    # Improvement 2: filter |cp| > 4000 (near-mate noise, ~0 gradient)
    num_workers  = 4,
    weight_decay = 1e-4,
    grad_clip    = 1.0,     # max gradient norm; prevents AMP instability
    qat          = True,    # Improvement 1: QAT (fake-quant FT+hidden+activations)
)

# ── Cross-run checkpoint sync (Kaggle API) ────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault('KAGGLE_API_TOKEN', UserSecretsClient().get_secret('KAGGLE_API_TOKEN'))
except Exception as e:
    print(f'no KAGGLE_API_TOKEN secret ({e}) -- checkpoint sync disabled, training from scratch')

def _kaggle(args):
    return _sp2.run(['kaggle'] + args, capture_output=True, text=True)

def ckpt_download():
    """Pull halfkav2.pt + resume_state.pt from CKPT_DATASET into WORK_DIR, if present."""
    if 'KAGGLE_API_TOKEN' not in os.environ:
        return
    print(f'Checking {CKPT_DATASET} for a previous checkpoint...')
    r = _kaggle(['datasets', 'download', '-d', CKPT_DATASET, '-p', str(WORK_DIR), '--unzip', '--force'])
    if r.returncode != 0:
        print('  no previous checkpoint found -- starting fresh')
        return
    found = [p.name for p in (OUT_PT, RESUME_PT) if p.exists()]
    print(f'  downloaded: {", ".join(found) if found else "(nothing usable)"}')

def ckpt_upload(message):
    """Push halfkav2.pt + resume_state.pt to CKPT_DATASET as a new version."""
    if 'KAGGLE_API_TOKEN' not in os.environ:
        return
    with _tmp2.TemporaryDirectory() as d:
        d = Path(d)
        for f in (OUT_PT, RESUME_PT):
            if f.exists():
                shutil.copy2(f, d / f.name)
        meta = {'title': 'eclipse-checkpoint', 'id': CKPT_DATASET, 'licenses': [{'name': 'other'}]}
        (d / 'dataset-metadata.json').write_text(_json2.dumps(meta))
        r = _kaggle(['datasets', 'version', '-p', str(d), '-m', message, '--dir-mode', 'tar'])
        if r.returncode != 0 and ('404' in (r.stderr or '') or 'not found' in (r.stderr or '').lower()):
            r = _kaggle(['datasets', 'create', '-p', str(d)])
        if r.returncode != 0:
            print(f'  checkpoint sync failed: {(r.stderr or r.stdout)[:300]}')
        else:
            print(f'  checkpoint synced to {CKPT_DATASET}: {message}')

ckpt_download()

# ── Device / GPU setup ────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory // 2**30} GB')
else:
    n_gpus = 0
    print('device: cpu')

# ── Binary preprocessing helpers ─────────────────────────────────────────────
_MAGIC2   = b'HKAV2BIN'
_RECSIZE2 = 32*2 + 32*2 + 2   # us(uint16×32) + them(uint16×32) + tgt(float16)
_BATCH2   = 50_000

def _prep_batch(args):
    # args = (lines, cp_scale, max_cp)
    lines, cp_scale, max_cp = args
    import math, numpy as np
    FT=45056; MA=32; NS=11
    _P={'p':1,'n':2,'b':3,'r':4,'q':5,'k':6}
    def _bd(b):
        p,sq=[],56
        for ch in b:
            if ch=='/': sq-=16; continue
            if ch.isdigit(): sq+=int(ch); continue
            p.append((sq,0 if ch.isupper() else 1,_P[ch.lower()])); sq+=1
        return p
    def _fi(ks,ps,pt,o):
        if o and pt==6: raise ValueError
        return ks*(64*NS)+ps*NS+((pt-1) if o else (5+pt-1))
    def _ff(fen):
        b,s=fen.split()[0],fen.split()[1]; pcs=_bd(b); sc=0 if s=='w' else 1
        ks=[None,None]
        for sq,c,pt in pcs:
            if pt==6: ks[c]=sq
        if None in ks: raise ValueError
        ix=[[],[]]
        for pp in(0,1):
            kq=ks[pp] if pp==0 else ks[pp]^56
            for sq,c,pt in pcs:
                if c==pp and pt==6: continue
                ix[pp].append(_fi(kq,sq if pp==0 else sq^56,pt,c==pp))
        return (ix[0],ix[1]) if sc==0 else (ix[1],ix[0])
    def _pl(line):
        line=line.strip()
        if not line or line.startswith('#'): return None
        p=line.split(';')
        if len(p)==4:
            try: fen=p[0].strip(); wp=float(p[1])+0.5*float(p[2])
            except: return None
        elif len(p)==2:
            try:
                fen=p[0].strip(); cp=float(p[1])
                if abs(cp) > max_cp: return None   # filter near-mate noise
                wp=1.0/(1.0+math.exp(-cp/cp_scale))
            except: return None
        else: return None
        try: us,them=_ff(fen)
        except: return None
        ua=np.full(MA,FT,dtype=np.uint16); ta=np.full(MA,FT,dtype=np.uint16)
        ua[:min(len(us),MA)]=us[:MA]; ta[:min(len(them),MA)]=them[:MA]
        return ua,ta,np.float16(wp)
    dt=np.dtype([('us',np.uint16,(MA,)),('them',np.uint16,(MA,)),('tgt',np.float16)])
    recs=[r for line in lines for r in [_pl(line)] if r]
    if not recs: return b''
    buf=np.empty(len(recs),dtype=dt)
    for i,(u,t,w) in enumerate(recs): buf[i]['us']=u; buf[i]['them']=t; buf[i]['tgt']=w
    return buf.tobytes()

def _batches2(path, skip, limit):
    op=_gz2.open if str(path).endswith('.gz') else open
    batch=[]; seen=0; taken=0
    with op(path,'rt',encoding='utf-8') as f:
        for line in f:
            s=line.strip()
            if s and not s.startswith('#'):
                if seen < skip: seen+=1; continue
                batch.append(line)
                if len(batch)==_BATCH2:
                    yield (batch, TRAIN_CFG['cp_scale'], TRAIN_CFG['max_cp'])
                    taken+=len(batch); batch=[]
                    if limit and taken>=limit: return
    if batch: yield (batch, TRAIN_CFG['cp_scale'], TRAIN_CFG['max_cp'])

def preprocess_chunk(skip, limit, out_path=None):
    """Preprocess up to `limit` positions starting at `skip` into out_path
    (defaults to BIN_PATH). Returns number of records written."""
    if out_path is None:
        out_path = BIN_PATH
    disk = shutil.disk_usage(WORK_DIR)
    need_gb = (limit * _RECSIZE2) / 1e9 + 1.5
    avail_gb = disk.free / 1e9
    if avail_gb < need_gb:
        raise RuntimeError(
            f'Not enough disk: need {need_gb:.1f} GB, have {avail_gb:.1f} GB free.')

    print(f'\n--- Preprocessing skip={skip:,} limit={limit:,}  '
          f'(disk free: {avail_gb:.1f} GB) ---')
    total=0; t0=_time2.time()
    with open(out_path,'wb') as fout:
        fout.write(_MAGIC2); fout.write(_struct2.pack('<q',0))
        with _mp.Pool(TRAIN_CFG['num_workers']) as pool:
            for data in pool.imap(_prep_batch, _batches2(DATA_PATH, skip, limit), chunksize=4):
                if not data: continue
                nr=len(data)//_RECSIZE2
                if limit and total+nr>limit: nr=limit-total; data=data[:nr*_RECSIZE2]
                fout.write(data); total+=nr
                el=_time2.time()-t0
                print(f'\r  {total:>12,}/{limit:,}  {total*_RECSIZE2/1e9:.1f} GB  '
                      f'{total/el/1e3 if el else 0:.0f}k pos/s', end='', flush=True)
                if limit and total>=limit: break
    with open(out_path,'r+b') as f: f.seek(8); f.write(_struct2.pack('<q',total))
    print(f'\n  done: {total:,} in {_time2.time()-t0:.0f}s')
    return total

# ── Premade chunk lookup ──────────────────────────────────────────────────
# If a dataset of pre-built HKAV2BIN chunks (eval_chunk_NN.bin, produced by
# scripts/preprocess_halfkav2.py --cp-scale 300 --max-cp 4000 to match
# TRAIN_CFG above) is attached under /kaggle/input, use it directly and skip
# FEN parsing for that chunk entirely. Falls back to preprocess_chunk() for
# any chunk_idx not covered by the premade set.
import glob, re
PREMADE_CHUNKS = {}
for _p in glob.glob('/kaggle/input/*/eval_chunk_*.bin'):
    _m = re.search(r'eval_chunk_(\d+)\.bin$', _p)
    if _m:
        PREMADE_CHUNKS[int(_m.group(1))] = Path(_p)
if PREMADE_CHUNKS:
    print(f'found {len(PREMADE_CHUNKS)} premade chunk(s): {sorted(PREMADE_CHUNKS)}')

def load_chunk(chunk_idx, skip, limit):
    """Use a premade chunk if available for chunk_idx, else preprocess live.
    Returns (bin_path, n_rec, created_locally)."""
    premade = PREMADE_CHUNKS.get(chunk_idx)
    if premade is not None:
        print(f'  Using premade chunk: {premade}')
        with open(premade, 'rb') as f:
            f.seek(8)
            n_rec = _struct2.unpack('<q', f.read(8))[0]
        return premade, n_rec, False
    if DATA_PATH is None:
        # No premade chunk for this index and no text source to live-preprocess
        # from => the premade set is exhausted. Report 0 records so the training
        # loop advances to the next epoch instead of crashing on a None source.
        print(f'  No premade chunk {chunk_idx} and no text source -- end of data.')
        return BIN_PATH, 0, False
    n_rec = preprocess_chunk(skip, limit)
    return BIN_PATH, n_rec, True

# ── Validation set from a premade chunk ───────────────────────────────────
# Premade chunks are built with the SAME cp_scale/max_cp as TRAIN_CFG, so the
# first VAL_SIZE records of chunk 0 are bit-identical to what
# preprocess_chunk(0, VAL_SIZE) produces from the text -- but without needing
# the 57GB eval_training.txt attached. Slice them straight out of the chunk.
def slice_val_from_premade(src, n_val, out_path):
    with open(src, 'rb') as f:
        assert f.read(8) == _MAGIC2, f'{src} is not an HKAV2BIN file'
        total = _struct2.unpack('<q', f.read(8))[0]
        n = min(n_val, total)
        payload = f.read(n * _RECSIZE2)
    with open(out_path, 'wb') as g:
        g.write(_MAGIC2); g.write(_struct2.pack('<q', n)); g.write(payload)
    return n

# ── Model ─────────────────────────────────────────────────────────────────────
base_net = HalfKAv2Net(qat=TRAIN_CFG['qat']).to(device)

_net2wider_applied = False
if OUT_PT.exists():
    print(f'Resuming weights from {OUT_PT}')
    sd = torch.load(OUT_PT, map_location=device)
    # One-time net2widernet transition: the checkpoint is still the narrow net,
    # so widen it (warm start) and train fresh. Once a wide checkpoint has been
    # saved, this branch no longer fires and training resumes normally.
    if NET2WIDER and 'ft.weight' in sd and sd['ft.weight'].shape == (FT_OUT_SRC, FT_IN_FEATURES):
        print(f'net2widernet: widening FT {FT_OUT_SRC}->{FT_OUT}, L1 {L1_OUT_SRC}->{L1_OUT}, '
              f'L2 {L2_OUT_SRC}->{L2_OUT} (function-preserving)')
        sd = net2wider_state_dict(sd)
        _net2wider_applied = True
    if 'ft.weight' in sd and sd['ft.weight'].shape == (FT_OUT, FT_IN_FEATURES):
        ft_w  = sd['ft.weight'].T
        pad   = torch.zeros(1, ft_w.shape[1], device=device)
        new_sd = {
            'ft.weight': torch.cat([ft_w, pad], dim=0),
            'ft_bias':   sd['ft.bias'],
            **{k: sd[k] for k in ('l1.weight','l1.bias','l2.weight','l2.bias',
                                   'l3.weight','l3.bias') if k in sd},
        }
        base_net.load_state_dict(new_sd)
    else:
        base_net.load_state_dict(sd)
else:
    print('No checkpoint found -- training from scratch')

# DataParallel — do NOT torch.compile; it breaks DataParallel replication on T4 x2
if n_gpus > 1:
    net = _nn.DataParallel(base_net)
    print(f'DataParallel: {n_gpus} GPUs, {TRAIN_CFG["batch_size"]//n_gpus} samples/GPU')
else:
    net = base_net

opt    = torch.optim.Adam(base_net.parameters(),
                          lr=TRAIN_CFG['lr'], weight_decay=TRAIN_CFG['weight_decay'])
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

_est_chunks = EPOCHS * ((1_600_000_000 + CHUNK_SIZE - 1) // CHUNK_SIZE)  # ~48
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, _est_chunks))

# ── Resume training position (epoch/chunk/optimizer/LR schedule) ─────────────
global_step = 0
start_epoch = 0
start_chunk = 0
if _net2wider_applied:
    print('net2widernet: fresh optimizer/schedule for the widened net')
if RESUME_PT.exists() and not _net2wider_applied:
    rs = torch.load(RESUME_PT, map_location='cpu')
    opt.load_state_dict(rs['optimizer'])
    cosine.load_state_dict(rs['cosine'])
    scaler.load_state_dict(rs['scaler'])
    global_step = rs['global_step']
    start_epoch = rs['epoch']
    start_chunk = rs['chunk_idx'] + 1
    print(f'Resuming training position: epoch {start_epoch} chunk {start_chunk}  '
          f'(global_step {global_step:,})')

# ── Fixed validation set (created once, reused every chunk / epoch) ───────────
print('\nCreating fixed validation set (200k positions)...')
_val_src = PREMADE_CHUNKS.get(0)
if _val_src is not None:
    _nval = slice_val_from_premade(_val_src, VAL_SIZE, VAL_BIN)
    print(f'  sliced {_nval:,} val records from premade {_val_src.name}')
elif DATA_PATH is not None:
    preprocess_chunk(0, VAL_SIZE, VAL_BIN)
else:
    raise FileNotFoundError('No premade chunk 0 and no eval_training.txt to build val set')
print(f'val_fixed.bin ready ({VAL_BIN.stat().st_size/1e6:.1f} MB)')

# ── Training loop ─────────────────────────────────────────────────────────────
prev_named_ckpt = None
if start_epoch >= EPOCHS:
    print('\nTraining already complete (resume_state past EPOCHS).')

for epoch in range(start_epoch, EPOCHS):
    print(f'\n\n#################### EPOCH {epoch} ####################')
    current_skip = start_chunk * CHUNK_SIZE if epoch == start_epoch else 0
    chunk_idx    = start_chunk               if epoch == start_epoch else 0

    while True:
        print(f'\n=== Epoch {epoch}  Chunk {chunk_idx}  pos {current_skip:,}–{current_skip+CHUNK_SIZE:,} ===')
        chunk_bin, n_rec, created_locally = load_chunk(chunk_idx, current_skip, CHUNK_SIZE)
        if n_rec == 0:
            print('  Data exhausted — next epoch.')
            break

        # Fixed val set — same 200k positions every chunk so loss is comparable.
        val_ds   = HalfKAv2BinaryDataset(VAL_BIN)
        train_ds = HalfKAv2BinaryDataset(chunk_bin)

        loader = DataLoader(
            train_ds,
            batch_size=TRAIN_CFG['batch_size'],
            shuffle=True,
            num_workers=TRAIN_CFG['num_workers'],
            collate_fn=collate,
            drop_last=True,
            pin_memory=(device.type == 'cuda'),
            persistent_workers=True,
            prefetch_factor=4,
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=TRAIN_CFG['batch_size'],
            shuffle=False,
            num_workers=2,
            collate_fn=collate,
            pin_memory=(device.type == 'cuda'),
            persistent_workers=False,
        )

        # ── Train ──
        net.train()
        ep_loss=0.0; nbatch=0; t_start=_time2.time()
        for us, them, target in loader:
            us     = us.to(device, non_blocking=True)
            them   = them.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)

            if global_step < TRAIN_CFG['warmup_steps']:
                scale = (global_step + 1) / TRAIN_CFG['warmup_steps']
                for g in opt.param_groups: g['lr'] = TRAIN_CFG['lr'] * scale

            with torch.autocast(device_type='cuda', dtype=torch.float16,
                                 enabled=(device.type == 'cuda')):
                logits = net(us, them)
                loss   = F.binary_cross_entropy_with_logits(
                             logits.squeeze(1), target.squeeze(1))

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(base_net.parameters(), TRAIN_CFG['grad_clip'])
            scaler.step(opt)
            scaler.update()

            with torch.no_grad():
                base_net.ft.weight[FT_IN_FEATURES].zero_()

            ep_loss+=loss.item(); nbatch+=1; global_step+=1
            if nbatch % 500 == 0:
                pos_s = nbatch * TRAIN_CFG['batch_size'] / (_time2.time() - t_start)
                print(f'  step {global_step:,}  loss {loss.item():.5f}  {pos_s/1e3:.0f}k pos/s')

        # ── Validate on fixed set ──
        net.eval()
        v_sum=0.0; v_n=0
        with torch.no_grad():
            for us, them, target in val_loader:
                us=us.to(device,non_blocking=True)
                them=them.to(device,non_blocking=True)
                target=target.to(device,non_blocking=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16,
                                    enabled=(device.type=='cuda')):
                    logits=net(us,them)
                v_sum+=F.binary_cross_entropy_with_logits(
                    logits.squeeze(1),target.squeeze(1)).item()*target.shape[0]
                v_n+=target.shape[0]
        val_loss = v_sum / max(1, v_n)

        cosine.step()
        elapsed = _time2.time() - t_start
        print(f'\n  Epoch {epoch} Chunk {chunk_idx}: '
              f'train={ep_loss/max(1,nbatch):.5f}  val={val_loss:.5f}  '
              f'lr={cosine.get_last_lr()[0]:.2e}  time={elapsed/60:.1f}min')

        # ── Save checkpoint + training position, sync to Kaggle ──
        named_ckpt = WORK_DIR / f'halfkav2_epoch{epoch}_chunk{chunk_idx}.pt'
        torch.save(base_net.export_state_dict_for_quantization(), named_ckpt)
        torch.save(base_net.export_state_dict_for_quantization(), OUT_PT)
        torch.save(dict(
            epoch=epoch, chunk_idx=chunk_idx, global_step=global_step,
            optimizer=opt.state_dict(), cosine=cosine.state_dict(),
            scaler=scaler.state_dict(),
        ), RESUME_PT)
        if prev_named_ckpt is not None and prev_named_ckpt.exists():
            prev_named_ckpt.unlink()
        prev_named_ckpt = named_ckpt
        print(f'  Saved {named_ckpt.name}')
        ckpt_upload(f'epoch{epoch} chunk{chunk_idx} step{global_step:,} val={val_loss:.5f}')

        del loader, val_loader, train_ds, val_ds
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        if created_locally and BIN_PATH.exists():
            BIN_PATH.unlink()
        print(f'  disk free: {shutil.disk_usage(WORK_DIR).free/1e9:.1f} GB')

        current_skip += CHUNK_SIZE
        chunk_idx    += 1

print('\nAll done!')


## Saving the artifact off Kaggle

1. Click **Save Version → Save & Run All**. The notebook output (including `/kaggle/working/halfkav2.pt`) is captured on the version that finishes successfully.
2. Open the saved version → **Output** tab → click `halfkav2.pt` → Download.
3. Locally, drop it into `data/` and run the convert command above. See `KAGGLE.md` for verification steps and saturation-warning interpretation.